# Out-of-Sample Test: 2023-2026

In [ ]:
import sys, os, pickle
sys.path.insert(0, '.')          # find env.py, evaluate.py, plots.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import joblib
import warnings
warnings.filterwarnings('ignore')

from stable_baselines3 import PPO
from evaluate import sharpe, max_drawdown

%matplotlib inline
plt.rcParams.update({'figure.dpi':110, 'axes.grid':True,
                     'grid.alpha':0.3, 'axes.spines.top':False,
                     'axes.spines.right':False})

TICKER_A         = os.environ.get('TE_A', 'V')
TICKER_B         = os.environ.get('TE_B', 'MA')
OBS_MODE         = os.environ.get('TE_OBS', 'zadj')
SEED             = os.environ.get('TE_SEED', '8')
PAIR_NAME = f'{TICKER_A}_{TICKER_B}'

TEST_START       = '2023-01-01'   # after training window ends
TEST_END         = '2026-06-01'
LOOKBACK_START   = '2022-07-01'

BETA        = float(np.load(f'models_{PAIR_NAME}_real/beta.npy')[0])
print(f'Beta loaded: {BETA:.4f}')
ZSCORE_WINDOW    = 20
TRANSACTION_COST = 0.001
INITIAL_CAPITAL  = 10_000.0

_ou = pickle.load(open(f'regime_output_{PAIR_NAME}/ou_params.pkl', 'rb'))
MU_R       = _ou['mu_r']
SIGMA_R    = _ou['sigma_r']
KAPPA      = _ou['kappa']
STAT_STD_R = {r: float(SIGMA_R[r] / np.sqrt(2.0 * KAPPA)) for r in range(3)}
CALIB_ALPHA = 0.05   # online calibration EMA rate (must match the saved models' training)

REGIME_COLORS = ['#2196F3', '#FF9800', '#E91E63']
REGIME_NAMES  = ['Stable', 'Neutral', 'Crisis']

print(f'Pair {PAIR_NAME}  OBS_MODE={OBS_MODE}  seed={SEED}')
print(f'Test window {TEST_START} -> {TEST_END}')
print(f'MU_R={ {r: round(MU_R[r],4) for r in range(3)} }  KAPPA={KAPPA:.2f}')
print(f'STAT_STD_R={ {r: round(STAT_STD_R[r],4) for r in range(3)} }')
print('Config OK')

## 1. Download Real Test Data

In [ ]:
raw = yf.download(
    [TICKER_A, TICKER_B],
    start  = LOOKBACK_START,
    end    = TEST_END,
    auto_adjust = True,
    progress    = False
)['Close'].dropna()

log_A_full = np.log(raw[TICKER_A].values)
log_B_full = np.log(raw[TICKER_B].values)
dates_full = raw.index

from statsmodels.tools import add_constant
from statsmodels.regression.linear_model import OLS

lookback_mask = dates_full < pd.Timestamp(TEST_START)
log_A_lb = log_A_full[lookback_mask]
log_B_lb = log_B_full[lookback_mask]

stored_beta = float(np.load(f'models_{PAIR_NAME}_real/beta.npy')[0])
_fit      = OLS(log_A_lb, add_constant(log_B_lb)).fit()
INTERCEPT = float(_fit.params[0])   # cointegration intercept (alpha)
BETA      = float(_fit.params[1])
print(f'Beta re-estimated from lookback window : {BETA:.4f}')
print(f'Intercept (alpha)                      : {INTERCEPT:.4f}')
print(f'Stored training beta (for reference)   : {stored_beta:.4f}')

spread_full = log_A_full - BETA * log_B_full

test_start_idx = np.searchsorted(
    dates_full, pd.Timestamp(TEST_START)
)

## 2. Test-Window Log Prices

In [ ]:
spread_test = spread_full[test_start_idx:]
dates_test  = dates_full[test_start_idx:]
T_test      = len(spread_test)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(dates_test, log_A_full[test_start_idx:], color='#2196F3', lw=1.3,
        label=f'log {TICKER_A}')
# Fitted leg: alpha + beta*log(B) is the regression's prediction of log(A),
# so the two lines sit on top of each other and the gap between them is the spread.
ax.plot(dates_test, INTERCEPT + BETA * log_B_full[test_start_idx:],
        color='#E91E63', lw=1.3,
        label=f'{INTERCEPT:.2f} + {BETA:.2f}·log {TICKER_B}')
ax.set_ylabel('Log price')
ax.set_title(f'Test-Window Log Prices — {TICKER_A} vs fitted {BETA:.2f}·log {TICKER_B}')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Evaluate on the full test window.
EVAL_START, EVAL_END = 0, T_test
EVAL_OFFSET = EVAL_START
spread_eval = spread_test[EVAL_START:EVAL_END]
dates_eval  = dates_test[EVAL_START:EVAL_END]
print(f'Eval window: {dates_eval[0].date()} -> {dates_eval[-1].date()}  ({len(spread_eval)} days)')

## 3. Load HMM

In [ ]:
# Load HMM and feature scaler from generator notebook
best_hmm   = joblib.load(f'regime_output_{PAIR_NAME}/hmm_model.pkl')
scaler     = np.load(f'regime_output_{PAIR_NAME}/feat_scaler.npy')
feat_means = scaler[0]
feat_stds  = scaler[1]
print('HMM loaded')

## 4. Online Regime Inference

In [ ]:
from scipy.stats import multivariate_normal as _mvn

def get_stationary(trans_matrix):
    from numpy.linalg import eig
    eigvals, eigvecs = eig(trans_matrix.T)
    idx  = np.argmin(np.abs(eigvals - 1.0))
    stat = np.real(eigvecs[:, idx])
    return (stat / stat.sum()).astype(np.float32)

stationary_dist = get_stationary(best_hmm.transmat_)
print(f'Stationary distribution: {stationary_dist}')


def _hmm_filter_step(alpha, feat_sc):
    """
    One step of the HMM forward filter.
    Uses only past observations — causal, no look-ahead.
    Returns normalised posterior [p_stable, p_neutral, p_crisis].
    """
    alpha_pred = best_hmm.transmat_.T @ alpha
    log_likes  = np.array([
        _mvn.logpdf(feat_sc, mean=best_hmm.means_[k], cov=best_hmm.covars_[k])
        for k in range(3)
    ])
    log_likes -= log_likes.max()
    likes      = np.exp(log_likes)
    alpha_new  = alpha_pred * likes
    total      = alpha_new.sum()
    return (alpha_new / total).astype(np.float32) if total > 1e-12 else alpha


def compute_features(spread_history, t,
                      window=ZSCORE_WINDOW, rev_window=60, rev_threshold=1.0):
    """3-feature vector at step t using only spread_history[:t+1]."""
    lo = max(0, t - window)
    w  = spread_history[lo: t + 1]
    if len(w) < 3:
        return None
    z      = (w[-1] - w.mean()) / (w.std() + 1e-8)
    rv     = np.diff(w).std() * np.sqrt(252) if len(w) > 1 else 1e-6
    logvol = np.log(max(rv, 1e-6))
    lo_r   = max(0, t - rev_window)
    w_r    = spread_history[lo_r: t + 1]
    if len(w_r) < 10:
        rate = 0.5
    else:
        mu_r  = w_r.mean(); std_r = w_r.std() + 1e-8
        zs    = (w_r[:-1] - mu_r) / std_r
        exc   = np.abs(zs) > rev_threshold
        rev   = (np.diff(w_r) * (mu_r - w_r[:-1])) > 0
        n_sig = exc.sum()
        n_rev = (exc & rev).sum()
        rate  = n_rev / n_sig if n_sig > 0 else 0.5
    logit_r = np.log(np.clip(rate, 0.05, 0.95) / (1 - np.clip(rate, 0.05, 0.95)))
    return np.array([z, logvol, logit_r], dtype=np.float32)


def get_zscore(spread_history, t, window=ZSCORE_WINDOW):
    lo = max(0, t - window)
    w  = spread_history[lo: t]
    if len(w) < 2:
        return 0.0
    return float(np.clip(
        (spread_history[t] - w.mean()) / (w.std() + 1e-8), -5.0, 5.0
    ))


def get_vol(spread_history, t, window=ZSCORE_WINDOW):
    lo  = max(0, t - window)
    w   = spread_history[lo: t + 1]
    if len(w) < 2:
        return 0.0
    return float(np.clip(np.diff(w).std() * np.sqrt(252), 0.0, 3.0))


print('Warming up forward filter on lookback period...')
alpha = stationary_dist.copy()
for t in range(test_start_idx):   # entire pre-test history
    feat_raw = compute_features(spread_full, t)
    if feat_raw is not None:
        feat_sc = (feat_raw - feat_means) / (feat_stds + 1e-8)
        alpha   = _hmm_filter_step(alpha, feat_sc)
print(f'Filter state after warm-up: '
      f'p_stable={alpha[0]:.3f}  p_neutral={alpha[1]:.3f}  p_crisis={alpha[2]:.3f}')

print('Computing regime probabilities for test period...')
regimes_soft_live = np.zeros((T_test, 3), dtype=np.float32)

for t in range(T_test):
    feat_raw = compute_features(spread_full, test_start_idx + t)
    if feat_raw is not None:
        feat_sc = (feat_raw - feat_means) / (feat_stds + 1e-8)
        alpha   = _hmm_filter_step(alpha, feat_sc)
    regimes_soft_live[t] = alpha


regimes_live = np.argmax(regimes_soft_live, axis=1).astype(np.int8)

unique, counts = np.unique(regimes_live, return_counts=True)
print('Regime distribution (argmax of posteriors):')
for r, c in zip(unique, counts):
    print(f'  {REGIME_NAMES[r]:8s}: {c:4d} days  ({c/T_test*100:.1f}%)')
print()
print('Mean posterior probabilities:')
print(f'  p_stable : {regimes_soft_live[:,0].mean():.3f}')
print(f'  p_neutral: {regimes_soft_live[:,1].mean():.3f}')
print(f'  p_crisis : {regimes_soft_live[:,2].mean():.3f}')

## 5. Strategy Rollout Helpers

In [ ]:
from evaluate import ZScorePolicy

def run_strategy(obs_fn, action_fn,
                 spread, regimes, dates,
                 tc=TRANSACTION_COST,
                 initial_capital=INITIAL_CAPITAL):
    T        = len(spread)
    capital  = initial_capital
    position = 0.0
    capital_hist  = [capital]
    position_hist = []
    reward_hist   = []
    trade_hist    = []

    for t in range(1, T):
        obs    = obs_fn(t, position)
        w_new  = float(np.clip(action_fn(obs), -1.0, 1.0))
        dz     = float(spread[t] - spread[t - 1])
        pnl    = position * dz
        cost   = tc * abs(w_new - position)
        reward = pnl - cost
        dollar = capital * reward / (1.0 + BETA)
        capital = max(capital + dollar, 1.0)
        traded  = abs(w_new - position) > 1e-4
        position = w_new
        capital_hist.append(capital)
        position_hist.append(position)
        reward_hist.append(reward)
        trade_hist.append(traded)

    return {
        'capital'   : np.array(capital_hist),
        'positions' : np.array(position_hist),
        'rewards'   : np.array(reward_hist),
        'n_trades'  : int(np.sum(trade_hist)),
        'total_ret' : (capital_hist[-1] / capital_hist[0] - 1) * 100,
        'sharpe'    : sharpe(pd.Series(capital_hist).pct_change().dropna().values),
        'max_dd'    : max_drawdown(capital_hist),
    }


def obs_baseline(t, position):
    """Baseline: [zscore, position, vol] — 3-dim."""
    idx = test_start_idx + EVAL_OFFSET + t
    return np.array([get_zscore(spread_full, idx), position,
                     get_vol(spread_full, idx)], dtype=np.float32)


def action_ppo(model):
    def _fn(obs):
        action, _ = model.predict(obs.reshape(1, -1), deterministic=True)
        return float(action[0])
    return _fn

def action_zscore(threshold=1.5):
    pol = ZScorePolicy(threshold=threshold)
    def _fn(obs):
        w, _ = pol.predict(np.asarray(obs, dtype=np.float32))
        return float(w[0])
    return _fn


regimes_eval = regimes_live[EVAL_START : EVAL_END]

## 6. Multi-Seed Ensemble - All Trained Seeds

In [ ]:
import glob, re

def load_seed_models(label):
    """All per-seed models under models_{PAIR}/{label}/seed*. Returns [(seed, model), ...]."""
    seed_dirs = sorted(glob.glob(f'models_{PAIR_NAME}_real/{label}/seed*'),
                       key=lambda d: int(re.search(r'seed(\d+)', d).group(1)))
    out = []
    for d in seed_dirs:
        s     = int(re.search(r'seed(\d+)', d).group(1))
        best  = f'{d}/best_model.zip'
        final = f'{d}/final.zip'
        path  = (best if os.path.exists(best) else final).replace('.zip', '')
        m = PPO.load(path); m.verbose = 0
        out.append((s, m))
    return out

ens_base = load_seed_models('baseline')
ens_reg  = load_seed_models('regime')
print(f'PPO Baseline : {len(ens_base)} seeds -> {[s for s, _ in ens_base]}')
print(f'PPO Regime   : {len(ens_reg)} seeds -> {[s for s, _ in ens_reg]}')


def make_obs_regime():
    """Regime-aware obs builder ([z_adj, position, vol]) with its OWN online-
    calibration state, so each seed's rollout starts from MU_R and is not
    contaminated by another seed's EMA history."""
    mu_live = {r: float(MU_R[r]) for r in range(3)}
    def _obs(t, position):
        idx   = test_start_idx + EVAL_OFFSET + t
        v     = get_vol(spread_full, idx)
        r     = int(regimes_live[EVAL_OFFSET + t])
        s_t   = float(spread_full[idx])
        mu_live[r] = CALIB_ALPHA * s_t + (1.0 - CALIB_ALPHA) * mu_live[r]
        z_adj = float(np.clip((s_t - mu_live[r]) / (STAT_STD_R[r] + 1e-8), -5.0, 5.0))
        return np.array([z_adj, position, v], dtype=np.float32)
    return _obs

def seed_curves_and_metrics(ens, regime):
    """Roll out each seed once; return (curves, metrics_df).
      curves  -- (n_seeds, T) capital curves indexed to 100
      metrics -- per-seed DataFrame [Return %, Sharpe, Max DD %, Trades]"""
    curves, rows = [], []
    for s, m in ens:
        obs_fn = make_obs_regime() if regime else obs_baseline   # fresh state per seed
        res = run_strategy(obs_fn, action_ppo(m),
                           spread_eval, regimes_eval, dates_eval)
        curves.append(res['capital'] / res['capital'][0] * 100)
        rows.append({'Return %': res['total_ret'], 'Sharpe': res['sharpe'],
                     'Max DD %': res['max_dd'], 'Trades': res['n_trades']})
    return np.array(curves), pd.DataFrame(rows, index=[s for s, _ in ens])

curves_base, metrics_base = seed_curves_and_metrics(ens_base, regime=False)
curves_reg,  metrics_reg  = seed_curves_and_metrics(ens_reg,  regime=True)

r_z     = run_strategy(obs_baseline, action_zscore(1.5),
                       spread_eval, regimes_eval, dates_eval)
curve_z = r_z['capital'] / r_z['capital'][0] * 100


def plot_ensemble(band, title):
    """band: 'std' -> mean +/- 1 SD across seeds; 'minmax' -> seed min-max."""
    fig, ax = plt.subplots(figsize=(13, 6))

    # Light regime shading
    prev_reg, t0 = int(regimes_eval[0]), dates_eval[0]
    for t in range(1, len(regimes_eval)):
        reg = int(regimes_eval[t])
        if reg != prev_reg or t == len(regimes_eval) - 1:
            ax.axvspan(t0, dates_eval[min(t, len(dates_eval) - 1)],
                       color=REGIME_COLORS[prev_reg], alpha=0.07)
            t0, prev_reg = dates_eval[min(t, len(dates_eval) - 1)], reg

    def draw(curves, color, label, ls='-'):
        mean = curves.mean(axis=0)
        if band == 'std':
            sd = curves.std(axis=0)
            lo, hi = mean - sd, mean + sd
        else:  # 'minmax'
            lo, hi = curves.min(axis=0), curves.max(axis=0)
        x = dates_eval[:len(mean)]
        ax.fill_between(x, lo, hi, color=color, alpha=0.18)
        ax.plot(x, mean, color=color, lw=2.0, ls=ls, label=label)

    draw(curves_base, '#2196F3', f'PPO Baseline  (n={len(ens_base)} seeds)')
    draw(curves_reg,  '#E91E63', f'PPO Regime  (n={len(ens_reg)} seeds)')
    ax.plot(dates_eval[:len(curve_z)], curve_z,
            color='#FF9800', lw=2.0, ls='--', label='Z-Score')

    ax.axhline(100, color='gray', lw=0.8, ls=':', label='Starting capital')
    ax.set_xlabel('Date', fontsize=20)
    ax.set_ylabel('Capital (indexed to 100)', fontsize=20)
    ax.tick_params(axis='both', labelsize=13)
    ax.legend(fontsize=16, loc='upper left')
    plt.tight_layout()
    plt.show()

plot_ensemble('std',    'band = mean +- 1 SD over seeds')
plot_ensemble('minmax', 'band = seed min-max')

# Across-seed summary of FINAL capital per PPO strategy
print(f"{'Strategy':<16}  {'Final (mean)':>12}  {'SD':>7}  {'Seed min':>9}  {'Seed max':>9}")
print('-' * 60)
for name, curves in [('PPO Baseline', curves_base), ('PPO Regime', curves_reg)]:
    finals = curves[:, -1]
    print(f"  {name:<14}  {finals.mean():>12.1f}  {finals.std():>7.1f}  "
          f"{finals.min():>9.1f}  {finals.max():>9.1f}")
print(f"  {'Z-Score':<14}  {curve_z[-1]:>12.1f}  {'--':>7}  {'--':>9}  {'--':>9}")

In [ ]:
prec = {'Return %': 2, 'Sharpe': 3, 'Max DD %': 2, 'Trades': 1}

def mean_se(metrics):
    m, se = metrics.mean(), metrics.sem()
    return pd.Series({c: f'{m[c]:.{prec[c]}f} ± {se[c]:.{prec[c]}f}' for c in prec})

z_row = pd.Series({c: f'{v:.{prec[c]}f}' for c, v in {
    'Return %': r_z['total_ret'], 'Sharpe': r_z['sharpe'],
    'Max DD %': r_z['max_dd'], 'Trades': float(r_z['n_trades'])}.items()})

summary = pd.DataFrame({
    f'PPO Baseline (n={len(ens_base)})': mean_se(metrics_base),
    f'PPO Regime (n={len(ens_reg)})':    mean_se(metrics_reg),
    'Z-Score (deterministic)':           z_row,
}).T[list(prec)]

display(summary)

print('\nPer-seed PPO Regime metrics:')
display(metrics_reg.round(prec))
print('Per-seed PPO Baseline metrics:')
display(metrics_base.round(prec))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compute rolling z-score on the test spread
W = 20
z_scores = np.full(len(spread_eval), np.nan)
for t in range(W, len(spread_eval)):
    w = spread_eval[t-W:t]
    z_scores[t] = (spread_eval[t] - w.mean()) / (w.std() + 1e-8)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(dates_eval, z_scores, color='black', lw=0.8, label='z-score')
ax.axhline( 1.5, color='red',  lw=1.2, ls='--', label='z-score = ±1.5 (entry)')
ax.axhline(-1.5, color='red',  lw=1.2, ls='--')
ax.axhline( 0.5, color='blue', lw=1.0, ls='--', label='z-score = ±0.5 (exit)')
ax.axhline(-0.5, color='blue', lw=1.0, ls='--')
ax.axhline(0,    color='gray',  lw=0.8, ls='-', alpha=0.5)
ax.set_ylabel('z-score', fontsize=20)
ax.set_xlabel('Date', fontsize=20)
ax.legend(fontsize=16)
plt.tight_layout()
plt.show()

## 7. Positions Over Time

In [ ]:
from matplotlib.lines import Line2D

pos_base = np.array([
    run_strategy(obs_baseline, action_ppo(m),
                 spread_eval, regimes_eval, dates_eval)['positions']
    for _, m in ens_base])
pos_reg = np.array([
    run_strategy(make_obs_regime(), action_ppo(m),
                 spread_eval, regimes_eval, dates_eval)['positions']
    for _, m in ens_reg])
pos_z = run_strategy(obs_baseline, action_zscore(1.5),
                     spread_eval, regimes_eval, dates_eval)['positions']

x = dates_eval[1:]


def shade_regimes(ax):
    """Light background shading by live HMM regime (matches plot_ensemble)."""
    prev_reg, t0 = int(regimes_eval[0]), dates_eval[0]
    for t in range(1, len(regimes_eval)):
        reg = int(regimes_eval[t])
        if reg != prev_reg or t == len(regimes_eval) - 1:
            ax.axvspan(t0, dates_eval[min(t, len(dates_eval) - 1)],
                       color=REGIME_COLORS[prev_reg], alpha=0.07)
            t0, prev_reg = dates_eval[min(t, len(dates_eval) - 1)], reg


fig, (ax0, ax1, ax2) = plt.subplots(3, 1, figsize=(13, 10), sharex=True)


def ensemble_panel(ax, curves, color, title):
    mean = curves.mean(axis=0)
    ax.fill_between(x, curves.min(axis=0), curves.max(axis=0),
                    color=color, alpha=0.18, label='seed min–max')
    ax.plot(x, mean, color=color, lw=1.6, label='mean position')
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_ylim(-1.15, 1.15)
    ax.set_ylabel(f'{title}\nposition', fontsize=18)
    ax.tick_params(axis='both', labelsize=13)
    ax.legend(fontsize=15, loc='upper left', ncol=2)

shade_regimes(ax0); ensemble_panel(ax0, pos_base, '#2196F3', 'PPO Baseline')
shade_regimes(ax1); ensemble_panel(ax1, pos_reg,  '#E91E63', 'PPO Regime')

shade_regimes(ax2)
ax2.plot(x, pos_z, color='#FF9800', lw=1.4, label='position')
ax2.axhline(0, color='gray', lw=0.8, ls=':')
ax2.set_ylim(-1.15, 1.15)
ax2.set_ylabel('Z-Score\nposition', fontsize=18)
ax2.tick_params(axis='both', labelsize=13)
ax2.legend(fontsize=15, loc='upper left')
ax2.set_xlabel('Date', fontsize=18)

# Shared regime legend on the top panel
reg_handles = [Line2D([0], [0], color=REGIME_COLORS[r], lw=6, alpha=0.4,
                      label=REGIME_NAMES[r]) for r in range(3)]
ax0.legend(handles=ax0.get_legend_handles_labels()[0] + reg_handles,
           fontsize=15, loc='upper left', ncol=5)

plt.tight_layout()
plt.show()

# Quick numeric summary of how each strategy is positioned
print(f"{'Strategy':<14}  {'mean pos':>9}  {'% long':>7}  {'% short':>8}  {'% flat':>7}")
print('-' * 54)
def _pos_stats(p):
    p = np.asarray(p).ravel()
    return (p.mean(), (p > 0.05).mean() * 100, (p < -0.05).mean() * 100,
            (np.abs(p) <= 0.05).mean() * 100)
for name, p in [('PPO Baseline', pos_base.mean(axis=0)),
                ('PPO Regime',   pos_reg.mean(axis=0)),
                ('Z-Score',      pos_z)]:
    mp, pl, ps, pf = _pos_stats(p)
    print(f'  {name:<12}  {mp:>9.3f}  {pl:>6.1f}%  {ps:>7.1f}%  {pf:>6.1f}%')